<a href="https://colab.research.google.com/github/adelmellit2013/Att_LSTM_Forecast_Sizing_PVEVCS_KSA/blob/main/AIoT_DRNN_PV_Forecast.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# This code was developed to forecat  PV power based on the past values of PV power
# P(t)=f(p(t-1),p(t-2),........) using LSTM, GRU, BiGRU and BiLSTM

import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
import pandas as pd
import seaborn as sns
from pylab import rcParams
import matplotlib.pyplot as plt
from matplotlib import rc
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.layers import Bidirectional, Dropout, Activation, Dense, LSTM, GRU
from tensorflow.keras.models import Sequential
%matplotlib inline
sns.set(style='whitegrid', palette='muted', font_scale=1.5)
rcParams['figure.figsize'] = 8, 6
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# PV power 50000 samples
data = pd.read_csv('/content/drive/MyDrive/powernew.csv')

In [ ]:
#data = df.sort_values('Datetime')
#df.head()
data.shape

In [ ]:
# Normalisation step
#-------------------
scaler = MinMaxScaler()
P = data.P.values.reshape(-1, 1)
scaled_IR = scaler.fit_transform(P)
scaled_IR.shape
np.isnan(scaled_IR).any()
scaled_IR = scaled_IR[~np.isnan(scaled_IR)]
scaled_IR = scaled_IR.reshape(-1, 1)
np.isnan(scaled_IR).any()

In [ ]:
##Processing

In [ ]:
##Processing
SEQ_LEN =9

def to_sequences(data, seq_len):
    d = []
    for index in range(len(data) - seq_len):
        d.append(data[index: index + seq_len])
    return np.array(d)

def preprocess(data_raw, seq_len, train_split):

    data = to_sequences(data_raw, seq_len)
    num_train = int(train_split * data.shape[0])
    X_train = data[:num_train, :-1, :]
    y_train = data[:num_train, -1, :]
    X_test = data[num_train:, :-1, :]
    y_test = data[num_train:, -1, :]
    return X_train, y_train, X_test, y_test


X_train, y_train, X_test, y_test = preprocess(scaled_IR, SEQ_LEN, train_split = 0.80)

In [ ]:
X_train.shape

In [ ]:
X_test.shape

In [ ]:
# Models

model = Sequential()

# Bi-LSTM
#model.add(Bidirectional(LSTM(units=100, input_shape=(X_train.shape[1], X_train.shape[2]))))

# Bi-GRU
 #model.add(Bidirectional(GRU(units=100, input_shape=(X_train.shape[1], X_train.shape[2]))))
# the true
model.add(Bidirectional(GRU(100)))


# GRU model
#model.add((GRU(units=100, input_shape=(X_train.shape[1], X_train.shape[2]))))

# LSTM model
#model.add((LSTM(units=100, input_shape=(X_train.shape[1], X_train.shape[2]))))

model.add(Dropout(rate=0.45))
model.add(Dense(units=12))
model.add(Dense(units=1))


In [ ]:
# Training the model
model.compile(loss='mean_squared_error', optimizer='adamax', metrics=['accuracy'])
history = model.fit( X_train, y_train, epochs=50, batch_size=64,verbose=1)

In [ ]:
# Model evaluation
#model.evaluate(X_test, y_test)
loss_history = history.history["loss"]
loss_ = pd.DataFrame(loss_history)
loss_.columns = ["loss BS=8"]
import numpy
numpy_loss_history = numpy.array(loss_history)
#numpy.savetxt("loss_history.txt", numpy_loss_history, delimiter=",")


In [ ]:
y_pred = model.predict(X_test)
y_train_inverse = scaler.inverse_transform(y_train)
y_test_inverse = scaler.inverse_transform(y_test)
y_pred_inverse = scaler.inverse_transform(y_pred)

import sklearn.metrics as metrics

mae = metrics.mean_absolute_error(y_test, y_pred)
mse = metrics.mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse) #mse**(0.5)
r2 = metrics.r2_score(y_test, y_pred)
r=np.sqrt(r2)
mape=np.mean(np.abs((y_test - y_pred) / y_test)) * 100

y_pred=np.array(y_pred)
m1=np.mean(y_pred)
y_test=np.array(y_test)
m2=np.mean(y_test)
mrpe=((m2-m1)/m2)*100
print("Results of sklearn.metrics:")
print("MAE:",mae)
print("MAPE:", mape)
print("RMSE:", rmse)
print("R-Squared:", r)
print("MRPE:", mrpe)


In [ ]:
%matplotlib inline
sns.set(style='whitegrid', palette='muted', font_scale=1)
rcParams['figure.figsize'] = 8, 6
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [ ]:
plt.plot(history.history['loss'])
#plt.plot(history.history['val_loss'])
plt.title('Model loss (MSE), Bi-GRU')
plt.ylabel('Loss (MSE)')
plt.xlabel('Epoch')
#plt.legend(['Train', 'Test'], loc='upper left')
plt.show()

In [ ]:
# Store the results
res_ = pd.DataFrame(y_pred_inverse)
res_.columns = ["Prediction"]

res = pd.DataFrame(y_test_inverse)
res.columns = ["Actual"]

path = '/content/drive/My Drive/outputLSTM.csv'
with open(path, 'w', encoding = 'utf-8-sig') as f:
  res_.to_csv(f)
path = '/content/drive/My Drive/outputLSTM1.csv'
with open(path, 'w', encoding = 'utf-8-sig') as f:
  res.to_csv(f)



In [ ]:
plt.plot(np.arange(0, len(y_train)), 2*y_train_inverse.flatten(), 'g', label="History")
plt.plot(np.arange(len(y_train), len(y_train) + len(y_test)), 2*y_test_inverse.flatten(), marker='.', label="Actual")
plt.plot(np.arange(len(y_train), len(y_train) + len(y_test)), 2*y_pred_inverse.flatten(), 'r', label="Predicted")
plt.title('BiGRU model')
plt.ylabel('PV power (W)')
plt.xlabel('Sample (1 min)')
plt.legend()
plt.show();


In [ ]:
plt.plot(y_test_inverse.flatten(), marker='.', label="Measured")
plt.plot(y_pred_inverse.flatten(), 'r', label="Predicted")
plt.title('BiGRU model')
plt.ylabel('PV power (W)')
plt.xlabel('Sample (1 min)')
plt.legend()
plt.show();

In [ ]:

plt.scatter(y_pred_inverse,y_test_inverse, s=4, color='black')
plt.title('BiGRU model')
plt.ylabel("Measured PV power (W)")
plt.xlabel("Predicted PV power (W)")
plt.show();


